# FIT5196 A1 — WIP notebook (Jasmine) · Task 2: the six standardised tables

**Group001** · Jasmine · work package **WP2** — rubric **B1, B2, B3**.

Personal working notebook (`01_WIP/`), **not** a deliverable. Section numbers match
`A1_solution_template.ipynb` verbatim (decision **D4**) so finished cells lift into
`00_Master/Group001_solution.ipynb` without renumbering.

**Sections owned here:** §4.1–4.6 (build the six tables) and §7 (export).

## 0. Configuration and reproducibility

All configurable paths live here. The notebook must run with **Restart and Run All**,
no manual edits, no network access.

Three requirements this section satisfies:

- `GROUP_ID`, `INPUT_DIR` and `OUTPUT_DIR` defined in one labelled place (Appendix A);
- **no student-specific absolute path** — a candidate list resolves whichever layout the
  notebook is run from, with the marker's layout first;
- personal runs write to a personal folder. Only a full run of
  `00_Master/Group001_solution.ipynb` writes to the shared `02_Outputs/` (**D9**).

In [1]:
# Colab only. Does nothing elsewhere, and nothing if the folder is not found.
try:
    from google.colab import drive
    from pathlib import Path as _Path
    import os as _os

    drive.mount('/content/drive')

    # Find the shared folder instead of hard-coding one member's Drive layout.
    _hit = next((p for p in _Path('/content/drive/MyDrive').rglob('Group001_A1/01_WIP')
                 if p.is_dir()), None)
    if _hit:
        _os.chdir(_hit)
        print('cwd:', _hit)
except Exception as e:
    print('Not on Colab, or Drive unavailable — using local paths.', e)

Mounted at /content/drive
cwd: /content/drive/MyDrive/Group001_A1/01_WIP


### 0.1 Environment and dependencies

Standard library plus pandas. Nothing needs installing on Colab.

The pandas version is printed because §8's reproducibility record needs it, and because one
behaviour §4 depends on is version-sensitive: `Series.astype(str)` turns `NaN` into the string
`'nan'` on pandas 2.x but leaves it as `NaN` on 3.x. Neither is the literal `'NaN'` sentinel
the specification requires, so §4 never relies on the cast to produce it.

In [2]:
# --- §0.1 Environment and dependencies ---

import json
import xml.etree.ElementTree as ET   # structured parsers — spec forbids regex on structure
import re                            # field values only, never document structure
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

print('pandas', pd.__version__)      # §8 needs the version; .astype(str) differs across majors

pandas 2.2.3


In [3]:
# --- §0 Configuration ---

GROUP_ID = 'Group001'

# First existing candidate wins. The marker's layout is first, so the submitted
# notebook works unchanged when they unzip and run.
INPUT_CANDIDATES = [
    Path('raw_input'),
    Path('Group001_A1/raw_input'),
    Path('../DATA/Group001_A1/raw_input'),
    Path('DATA/Group001_A1/raw_input'),
]
INPUT_DIR = next((p for p in INPUT_CANDIDATES if p.exists()), None)
assert INPUT_DIR is not None, f'No input folder found. Tried: {INPUT_CANDIDATES}'

JSON_PATH = INPUT_DIR / f'{GROUP_ID}_commerce.json'
XML_PATH  = INPUT_DIR / f'{GROUP_ID}_operations.xml'
DICT_PATH = INPUT_DIR.parent / 'public_data_dictionary.csv'

OUTPUT_DIR = Path('outputs_wip_jasmine')   # personal scratch only — D9
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Reading from :', INPUT_DIR.resolve())

Reading from : /content/drive/.shortcut-targets-by-id/1o61LcnA0kUG-ylT0Hv1KqaDDDTCBbXSC/Group001_A1/DATA/Group001_A1/raw_input


---

## 1. Parse and profile the two sources

> **WP1 — Echo.** Functions reproduced from `wip_echo_0825` so this notebook runs standalone.
> **Do not edit here.** Changes go to Echo; the master takes her version.

What the parser contract gives WP2 (DEC-011):

- `dict[str, DataFrame]` keyed by output-table name, column names already the dictionary's;
- **values are source-native, not normalised** — XML dates are still `DD/MM/YYYY`, money is
  still `'AUD 2,765.47'`. Normalising is §4's job, kept visible rather than hidden in the parser;
- three text fields keep a `_raw` suffix — the target is the *cleaned* version (DEC-014);
- every frame carries `source_system` at position 0. **Drop it before export.**

| | tables returned |
|---|---|
| `parse_json` | orders, order_items, deliveries, **customers**, product_reviews |
| `parse_xml` | orders, order_items, deliveries, **products**, product_reviews, *warehouses* |

`warehouses` is not an output table — never iterate `xml_tables` blindly.

In [4]:
# --- §1.1 JSON parser — reproduced from wip_echo_0825 (WP1). Do not edit here. ---

def to_snake(name, exceptions=None):
    """camelCase -> snake_case. 'ID' is treated as one word, so customerID -> customer_id."""
    exceptions = exceptions or {}
    if name in exceptions:
        return exceptions[name]
    name = name.replace("ID", "Id")
    name = re.sub(r"(?<!^)(?=[A-Z])", "_", name)   # insert _ before each inner capital
    return name.lower()


def parse_json(path, exceptions=None):
    """Read the commerce JSON export into flat tables with source-native values."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    def rename(record):
        return {to_snake(k, exceptions): v for k, v in record.items()}

    # One pass over orders fills three tables. append/extend is where the grain changes:
    # one header and one delivery per order, but many cart items.
    orders, order_items, deliveries = [], [], []
    for order in raw["orders"]:
        orders.append(rename(order["header"]))
        deliveries.append(rename(order["delivery"]))
        order_items.extend(rename(item) for item in order["shoppingCart"])

    tables = {
        "orders":          pd.DataFrame(orders),
        "order_items":     pd.DataFrame(order_items),
        "deliveries":      pd.DataFrame(deliveries),
        "customers":       pd.DataFrame([rename(c) for c in raw["customerProfiles"]]),
        "product_reviews": pd.DataFrame([rename(r) for r in raw["productReviews"]]),
    }
    for df in tables.values():
        df.insert(0, "source_system", "JSON")

    return tables, raw["exportMetadata"]

In [5]:
# --- §1.2 XML parser — reproduced from wip_echo_0825 (WP1). Do not edit here. ---

def element_to_record(element, exceptions=None):
    """One XML element -> one dict. Child tag becomes the column, child text the value."""
    exceptions = exceptions or {}
    # `child.text or ""`: an empty XML element gives None where JSON gives "". Without this
    # the two sources are not comparable and every blank reports a false difference. DEC-015.
    return {exceptions.get(child.tag, child.tag.lower()): (child.text or "")
            for child in element}


def parse_xml(path, exceptions=None):
    """Read the operations XML export into flat tables with source-native values."""
    root = ET.parse(path).getroot()

    orders, order_items, deliveries = [], [], []
    for order in root.findall("Orders/Order"):
        orders.append(element_to_record(order.find("Header"), exceptions))
        deliveries.append(element_to_record(order.find("Delivery"), exceptions))
        order_items.extend(element_to_record(item, exceptions)
                           for item in order.findall("Shopping_Cart/Item"))

    tables = {
        "orders":          pd.DataFrame(orders),
        "order_items":     pd.DataFrame(order_items),
        "deliveries":      pd.DataFrame(deliveries),
        "products":        pd.DataFrame([element_to_record(p, exceptions)
                                         for p in root.findall("ProductCatalogue/Product")]),
        "product_reviews": pd.DataFrame([element_to_record(r, exceptions)
                                         for r in root.findall("ProductReviews/Review")]),
        "warehouses":      pd.DataFrame([element_to_record(w, exceptions)
                                         for w in root.findall("WarehouseDirectory/Warehouse")]),
    }
    for df in tables.values():
        df.insert(0, "source_system", "XML")

    metadata = dict(root.attrib)
    metadata.update(element_to_record(root.find("Export_Metadata")))
    return tables, metadata

In [6]:
# --- §1.3 Run both parsers ---

# Exception tables are keyed on the ORIGINAL key/tag, so the two are not interchangeable.
# Passing the wrong one fails silently — it just misses and falls back to the general rule.
JSON_NAME_EXCEPTIONS = {
    'prior12MOrders': 'prior_12m_orders',   # the regex splits on capitals, not digits
    'customerNote':   'customer_note_raw',  # target field is the *clean* version
    'reviewText':     'review_body_raw',
}
XML_NAME_EXCEPTIONS = {
    'Customer_Note':       'customer_note_raw',
    'Review_Text':         'review_body_raw',
    'Product_Description': 'product_description_raw',
}

json_tables, json_meta = parse_json(JSON_PATH, JSON_NAME_EXCEPTIONS)
xml_tables,  xml_meta  = parse_xml(XML_PATH,  XML_NAME_EXCEPTIONS)

# Confirm the group allocation before trusting anything else (spec §3).
assert json_meta['groupAlias'] == GROUP_ID, json_meta
assert xml_meta['groupAlias']  == GROUP_ID, xml_meta

# Precondition for every concat in §4: the shared tables must line up by column name.
# This is the one failure mode the exception tables can cause silently.
BOTH_SOURCES = ['orders', 'order_items', 'deliveries', 'product_reviews']
for t in BOTH_SOURCES:
    j, x = set(json_tables[t].columns), set(xml_tables[t].columns)
    assert j == x, (t, sorted(j ^ x))

for source, tabs in (('JSON', json_tables), ('XML', xml_tables)):
    for name, df in tabs.items():
        print(f'{source:5s} {name:16s} {df.shape[0]:>7,} rows x {df.shape[1]:>2} cols')

JSON  orders             2,818 rows x 23 cols
JSON  order_items        8,826 rows x  7 cols
JSON  deliveries         2,818 rows x 21 cols
JSON  customers            500 rows x 21 cols
JSON  product_reviews    3,946 rows x 16 cols
XML   orders             2,818 rows x 23 cols
XML   order_items        8,833 rows x  7 cols
XML   deliveries         2,818 rows x 21 cols
XML   products           1,000 rows x 22 cols
XML   product_reviews    3,946 rows x 16 cols
XML   warehouses             3 rows x  4 cols


---

## 3. Text and regex functions

> **WP4 — Shawn.** Imported, never reimplemented here. Until G2 these are local placeholders
> that return the input unchanged (**D7**), which is what lets §4 be built before the real
> implementations exist. The banner below says which is in force.

Ten target fields across three of my tables come from three `_raw` columns:

| `_raw` column | table | target fields |
|---|---|---|
| `customer_note_raw` | `orders` | `customer_note_clean`, `promo_code` |
| `product_description_raw` | `products` | `product_description_clean` |
| `review_body_raw` | `product_reviews` | 7 fields |

So `order_items`, `deliveries` and `customers` are free of any WP4 dependency — which is why
they were built first.

> **→ SHAWN.** The signatures below are the ones §4 calls. If any differs from your module,
> §4 breaks at G2 rather than at your review — tell me now if the names or argument order
> are not these.

In [7]:
# --- §3 Text functions: import the published interface, or fall back to placeholders ---

TEXT_FN_CANDIDATES = [
    Path('Group001_text_functions.py'),
    Path('../00_Master/Group001_text_functions.py'),
]
_hit = next((p for p in TEXT_FN_CANDIDATES if p.exists()), None)

if _hit is not None:
    import sys
    sys.path.insert(0, str(_hit.parent))
    from Group001_text_functions import (
        clean_narrative_text, extract_order_reference, extract_product_sku,
        extract_promo_code, build_latin_analysis, contains_non_latin_script,
    )
    print('Text functions:', _hit)
else:
    # Deliberately wrong — pass-through — so any table built on them is obviously provisional.
    def clean_narrative_text(text):            return text
    def extract_order_reference(text):         return 'NaN'
    def extract_product_sku(text):             return 'NaN'
    def extract_promo_code(text):              return 'NaN'
    def build_latin_analysis(clean_text):      return clean_text
    def contains_non_latin_script(clean_text): return False


    print('!' * 74)
    print('!  WP4 module not found — using placeholders.')
    print('!  §4.1, §4.5 and §4.6 output is PROVISIONAL until G2.')
    print('!' * 74)

WP4_PLACEHOLDER = _hit is None

Text functions: Group001_text_functions.py


---

## 4. Build the six standardised relational tables

WP2 in full — rubric B1 (tables, grains, fields, field order, keys), B2 (field values and the
arithmetic chain), B3 (types, formats, missing-value sentinels).

### Pipeline order is a hard constraint


Building per source and unioning last fails *silently*. WP1 §1.3d shows why: JSON's reviews
reference 1,813 `order_item_id` values JSON does not contain, and XML's reference 1,724 that XML
does not. Neither export is referentially self-consistent, so every intermediate foreign-key
check on a per-source table would fail and look like a transformation bug.

No fan-out, no roll-up — source grain equals target grain for all six tables (WP1 §1.4). The only
cross-table calculation is `order_price`, summing `line_revenue` from `order_items`.

### Three source cases

| case | tables |
|---|---|
| both sources → concat, normalise, deduplicate | `orders`, `order_items`, `deliveries`, `product_reviews` |
| JSON only | `customers` |
| XML only | `products` |
| not an output table — dropped | `warehouses` |


**Cell order versus section numbering.** The headings follow the template's numbering verbatim
(D4), but the cells are ordered by dependency: `4.1 orders` derives `order_price` from
`4.2 order_items`, so 4.2 executes first. Restart and Run All is reproducible in this order; it
would not be in numeric order.

Build order was also chosen for cost: `order_items`, `deliveries` and `customers` carry no WP4
dependency and were built first to prove the pipeline; `orders` and `product_reviews` came last
because they carry nine of the ten derived text fields between them.

### 4.0 Target contract

B1 is assessed on filenames, grains, fields **and field order**; B3 on types, nullability and the
`NaN` sentinel. All of it is published in `public_data_dictionary.csv`, so the contract is *read*
rather than transcribed — transcription is where silent field-order and dtype errors come from.

The dictionary carries more than expected: `grain` per table, and a **`comparison_rule` per
field**.

> **→ YANDU.** Tolerances are declared, not ours to choose. Read `comparison_rule` per field
> rather than applying one rule per `data_type`: in `orders`, `coupon_discount`, `customer_lat`
> and `customer_long` are all `number` but compare **exact**, while only the four money fields
> carry `numeric tolerance 0.01`. A blanket 0.01 on latitude would pass errors of about a
> kilometre.
>
> **→ YANDU.** `product_reviews`' published grain is "one row per **canonical** review". The
> dictionary itself says the output is post-reconciliation, which settles whether §4 emits
> deduplicated tables. Worth a `DEC-` row.

In [8]:
# --- §4.0 Read the published contract ---

dd = pd.read_csv(DICT_PATH, keep_default_na=False)   # keep the literal 'NaN' sentinel visible

print('shape  ', dd.shape)
print('columns', dd.columns.tolist())
print()
print(dd['output_table'].value_counts().to_string())
dd.head(12)

shape   (111, 7)
columns ['output_table', 'grain', 'position', 'field_name', 'data_type', 'nullable', 'comparison_rule']

output_table
orders             23
product_reviews    21
products           21
deliveries         20
customers          20
order_items         6


,output_table,grain,position,field_name,data_type,nullable,comparison_rule
0,orders,one row per order,1,order_id,string,False,exact after published normalisation
1,orders,one row per order,2,source_system_record_id,string,False,exact after published normalisation
2,orders,one row per order,3,customer_id,string,False,exact after published normalisation
3,orders,one row per order,4,order_timestamp,datetime,False,exact after published normalisation
4,orders,one row per order,5,sales_channel,string,False,exact after published normalisation
5,orders,one row per order,6,payment_method,string,False,exact after published normalisation
6,orders,one row per order,7,currency,string,False,exact after published normalisation
7,orders,one row per order,8,nearest_warehouse,string,False,exact after published normalisation
8,orders,one row per order,9,order_status,string,False,exact after published normalisation
9,orders,one row per order,10,order_price,number,False,numeric tolerance 0.01


In [9]:
# --- §4.0 Contract lookups ---
# Built once here so there is one reading of the dictionary, not six.

OUTPUT_TABLES = ['orders', 'order_items', 'customers', 'deliveries', 'products', 'product_reviews']

def contract(table):
    """The published contract for one output table, straight from the dictionary."""
    rows = dd.loc[dd['output_table'] == table].sort_values('position')
    return {
        'grain':      rows['grain'].iloc[0],
        'fields':     rows['field_name'].tolist(),                    # position order -> B1
        'pk':         rows.loc[rows['position'] == 1, 'field_name'].iloc[0],
        'dtype':      dict(zip(rows['field_name'], rows['data_type'])),
        'nullable':   dict(zip(rows['field_name'], rows['nullable'])),
        'comparison': dict(zip(rows['field_name'], rows['comparison_rule'])),
    }

CONTRACT = {t: contract(t) for t in OUTPUT_TABLES}

for t in OUTPUT_TABLES:
    c = CONTRACT[t]
    print(f"{t:16s} {len(c['fields']):>2} fields · pk = {c['pk']:<14s} · {c['grain']}")

orders           23 fields · pk = order_id       · one row per order
order_items       6 fields · pk = order_item_id  · one row per order item
customers        20 fields · pk = customer_id    · one row per customer
deliveries       20 fields · pk = delivery_id    · one row per completed order
products         21 fields · pk = product_id     · one row per product
product_reviews  21 fields · pk = review_id      · one row per canonical review


In [10]:
# --- §4.0 The target contract, table by table ---
# Read from the dictionary rather than transcribed: transcription is where silent
# field-order and dtype errors come from.

def show_contract(table):
    c = CONTRACT[table]
    print(f"\n{table}  —  {c['grain']}  ({len(c['fields'])} fields, pk = {c['pk']})")
    rows = dd.loc[dd['output_table'] == table].sort_values('position')
    print(rows[['position', 'field_name', 'data_type', 'nullable', 'comparison_rule']]
          .to_string(index=False))

for t in OUTPUT_TABLES:
    show_contract(t)


orders  —  one row per order  (23 fields, pk = order_id)
 position              field_name data_type  nullable                     comparison_rule
        1                order_id    string     False exact after published normalisation
        2 source_system_record_id    string     False exact after published normalisation
        3             customer_id    string     False exact after published normalisation
        4         order_timestamp  datetime     False exact after published normalisation
        5           sales_channel    string     False exact after published normalisation
        6          payment_method    string     False exact after published normalisation
        7                currency    string     False exact after published normalisation
        8       nearest_warehouse    string     False exact after published normalisation
        9            order_status    string     False exact after published normalisation
       10             order_price    numbe

**One thing the whole group needs to settle.** Every one of `product_reviews`' 21 fields is
`nullable = False`, including four produced by text functions that return the literal `'NaN'`
when there is nothing to extract: `review_body_clean`, `review_body_latin_analysis`,
`extracted_order_reference`, `extracted_product_sku`.

Two readings:

- **A** — `nullable` means "the cell may be empty". `'NaN'` is three characters, so it is a
  value, and any string field may carry it.
- **B** — `nullable = True` marks the fields where the sentinel is permitted. Then
  `nullable = False` is a claim that the field always resolves on this data.

Evidence favours **B**: in `orders` the only two nullable fields are `coupon_code` and
`promo_code` — exactly the two that can legitimately be absent. If `nullable` only meant
non-empty there would be no reason to single those out.

**B is testable.** Once WP4 ships, assert that no `nullable = False` field contains `'NaN'`.
If one does, that is a genuine validation failure — which the rubric credits when it is
identified and treated, and penalises only if a value is fabricated to hide it.

> **→ SHAWN.** Your return values decide this. `clean_narrative_text` and `build_latin_analysis`
> return `'NaN'` when nothing readable remains, and both target fields are `nullable = False`.
> Once the real implementations land I will count how many rows land on the sentinel — if it is
> zero, both readings hold and the question disappears.
>
> **→ YANDU.** This is a `VAL-SCHEMA-` check, driven from the dictionary's `nullable` column
> rather than a hard-coded field list.

The plan derived from the dictionary is right for about 95% of fields. Two things it cannot
know, corrected below from the report rather than guessed:

- **derived fields have no source column** to normalise — `review_length_chars`,
  `review_word_count` and `contains_non_latin_script` are produced by WP4, not read from a file;
- **the dictionary has no `percent` type.** `coupon_discount` is `number` with an exact
  comparison rule, but the XML writes `'10%'`. Source spelling is not a contract fact.

> **→ ECHO.** Two notes on `NORMALISERS`.
>
> Your `percent` category is necessary and cannot be derived — the dictionary describes the
> target type, not how each file spells it. Worth saying so in that row's mapping text.
>
> `date` and `datetime` are separate `data_type` values and your map merges them. Parsing is the
> same, but output formatting is not: five fields are `date` (`signup_date`, `dispatch_date`,
> `promised_date`, `delivered_date`, `launch_date`, across three tables) and two are `datetime`
> (`order_timestamp`, `review_timestamp`). Treating all seven as timestamps appends `00:00:00`
> to the five, which B3 assesses. §4 follows the dictionary's two categories.
>
> Otherwise the plan I derived from the dictionary matches your hand-written map exactly — two
> independent routes to the same answer.

In [11]:
# --- §4.0 Derive the normalisation plan from the dictionary ---
# Money is not a data_type: the dictionary distinguishes it by comparison_rule, and date
# from datetime by data_type. Deriving the plan means it cannot drift from the contract.

MONEY_RULE = 'numeric tolerance 0.01'

def normalisation_plan(table):
    rows = dd.loc[dd['output_table'] == table]
    plan = {'money': [], 'number': [], 'date': [], 'datetime': [], 'boolean': []}
    for _, f in rows.iterrows():
        if f.data_type == 'number':
            plan['money' if f.comparison_rule == MONEY_RULE else 'number'].append(f.field_name)
        elif f.data_type in plan:
            plan[f.data_type].append(f.field_name)
    return {k: v for k, v in plan.items() if v}      # drop empty categories

PLAN = {t: normalisation_plan(t) for t in OUTPUT_TABLES}

for t in OUTPUT_TABLES:
    print(f'\n{t}')
    for kind, fields in PLAN[t].items():
        print(f'   {kind:9s} {fields}')


orders
   money     ['order_price', 'delivery_charges', 'tax_amount', 'order_total']
   number    ['coupon_discount', 'customer_lat', 'customer_long']
   datetime  ['order_timestamp']
   boolean   ['expedited_delivery']

order_items
   money     ['unit_price', 'line_revenue']
   number    ['quantity']

customers
   money     ['lifetime_value_before_period']
   number    ['prior_12m_orders']
   date      ['signup_date']
   boolean   ['marketing_consent']

deliveries
   money     ['delivery_cost']
   number    ['delay_days', 'fulfilment_hours', 'promised_days', 'tracking_event_count', 'shipping_distance_km', 'estimated_carbon_kg']
   date      ['dispatch_date', 'promised_date', 'delivered_date']
   boolean   ['on_time_in_full', 'signature_required']

products
   money     ['unit_price', 'unit_cost']
   number    ['launch_year', 'warranty_months', 'weight_kg']
   date      ['launch_date']
   boolean   ['recyclable_packaging', 'active_flag']

product_reviews
   number    ['rating', 'helpf

In [12]:
# --- §4.0 Correct the derived plan ---
# Two things the dictionary cannot express, written from the output above rather than
# guessed: which fields have no source column, and which source spells a number with '%'.

SOURCE_COLUMNS = {}
for tabs in (json_tables, xml_tables):
    for name, df in tabs.items():
        SOURCE_COLUMNS.setdefault(name, set()).update(df.columns)

PERCENT_FIELDS = {'coupon_discount'}   # XML writes '10%'; the contract only says 'number'

for table, plan in PLAN.items():
    for kind in list(plan):
        plan[kind] = [f for f in plan[kind] if f in SOURCE_COLUMNS.get(table, set())]
    plan['percent'] = [f for f in plan.pop('number', []) if f in PERCENT_FIELDS] or []
    plan['number']  = [f for f in normalisation_plan(table).get('number', [])
                       if f in SOURCE_COLUMNS.get(table, set()) and f not in PERCENT_FIELDS]
    PLAN[table] = {k: v for k, v in plan.items() if v}

for t in OUTPUT_TABLES:
    print(f'\n{t}')
    for kind, fields in PLAN[t].items():
        print(f'   {kind:9s} {fields}')


orders
   money     ['order_price', 'delivery_charges', 'tax_amount', 'order_total']
   datetime  ['order_timestamp']
   boolean   ['expedited_delivery']
   percent   ['coupon_discount']
   number    ['customer_lat', 'customer_long']

order_items
   money     ['unit_price', 'line_revenue']
   number    ['quantity']

customers
   money     ['lifetime_value_before_period']
   date      ['signup_date']
   boolean   ['marketing_consent']
   number    ['prior_12m_orders']

deliveries
   money     ['delivery_cost']
   date      ['dispatch_date', 'promised_date', 'delivered_date']
   boolean   ['on_time_in_full', 'signature_required']
   number    ['delay_days', 'fulfilment_hours', 'promised_days', 'tracking_event_count', 'shipping_distance_km', 'estimated_carbon_kg']

products
   money     ['unit_price', 'unit_cost']
   date      ['launch_date']
   boolean   ['recyclable_packaging', 'active_flag']
   number    ['launch_year', 'warranty_months', 'weight_kg']

product_reviews
   datetime  ['r

### 4.0.1 Normalisers

The parsers return source-native values by design, so every typed field needs converting before
it can be compared, calculated with, or written out. XML is the heavy side: every value arrives
as text. JSON is natively typed and needs only date parsing.

Values stay typed through §4 — Timestamps, floats, bools — and are formatted to the published
string forms only at export, in §7.

XML strings compare lexicographically and give wrong answers *silently*:
`'AUD 155.15' > 'AUD 1,827.30'` evaluates `True`, because `'5'` sorts after `','`. No sort,
comparison or aggregation happens before this step.

Booleans export as `True` / `False`, not `Y` / `N` or `1` / `0`. Stated as a
convention rather than left as an observed fact: the dictionary says only
`boolean`, and B3 assesses dtype conformance. The JSON writes `True` and the
XML writes `Y` for the same fact, so this is also why comparison must happen
after normalisation, not before.

In [13]:
# --- §4.0.1 Normalisers ---
# Values stay typed through §4 (Timestamps, floats, bools) and are formatted to the
# published string forms only at export, in §7.

def norm_money(value):
    """'AUD 2,765.47' or 2765.47 -> 2765.47. The thousands comma appears on only some
    values, so it is stripped rather than matched."""
    return round(float(str(value).replace('AUD', '').replace(',', '').strip()), 2)

def norm_percent(value):
    """'10%' or 10 -> 10.0. Percentage points, so arithmetic divides by 100."""
    return float(str(value).replace('%', '').strip())

def norm_bool(value):
    """'Y' / 'N' / True / False -> bool. Anything unrecognised becomes False silently,
    so §6 should assert that no boolean source column is ever blank."""
    return str(value).strip().lower() in {'y', 'yes', 'true'}

def norm_temporal(series, dayfirst):
    """Vectorised: pd.to_datetime already accepts a Series. Calling it per element pays
    the parser start-up cost on every row — same result, ~100x slower."""
    return pd.to_datetime(series.astype(str).str.strip(), dayfirst=dayfirst, errors='coerce')


def normalise_frame(df, table, dayfirst):
    """Apply the derived plan to one source's frame."""
    plan = PLAN.get(table, {})
    out = df.copy()
    for kind, fields in plan.items():
        for f in fields:
            if kind == 'money':            out[f] = out[f].map(norm_money)
            elif kind == 'percent':        out[f] = out[f].map(norm_percent)
            elif kind == 'boolean':        out[f] = out[f].map(norm_bool)
            elif kind in ('date', 'datetime'):
                out[f] = norm_temporal(out[f], dayfirst)
            elif kind == 'number':         out[f] = pd.to_numeric(out[f], errors='coerce')
    return out

In [14]:
# --- §4.0.1 Sanity check: normalise one table from both sources ---
# order_items is the smallest table with no derived fields, so it is the cheapest place
# to confirm the plan works before applying it to all six.

j = normalise_frame(json_tables['order_items'], 'order_items', dayfirst=False)
x = normalise_frame(xml_tables['order_items'],  'order_items', dayfirst=True)

print('dtypes after normalisation')
print(pd.DataFrame({'JSON': j.dtypes, 'XML': x.dtypes}).to_string())

print('\nsame row, both sources')
sid = sorted(set(j.order_item_id) & set(x.order_item_id))[0]
print(pd.DataFrame({
    'JSON': j.loc[j.order_item_id == sid].iloc[0],
    'XML':  x.loc[x.order_item_id == sid].iloc[0],
}).to_string())

dtypes after normalisation
                  JSON      XML
line_revenue   float64  float64
order_id        object   object
order_item_id   object   object
product_id      object   object
quantity         int64    int64
source_system   object   object
unit_price     float64  float64

same row, both sources
                      JSON          XML
line_revenue       1125.28      1125.28
order_id        HORD000015   HORD000015
order_item_id  HITM0000038  HITM0000038
product_id         PRD0389      PRD0389
quantity                 2            2
source_system         JSON          XML
unit_price          562.64       562.64


### 4.0.2 Combine and deduplicate

Deduplication handles **both** duplicate problems in one step: within-source repeats and
cross-source overlap. That is legitimate only because of two WP1 findings — within-source
duplicates are exactly two field-identical copies (A3), and shared keys agree on every field
once normalised (A5). The surviving row therefore depends on neither which copy nor which
source wins.

Expected row counts are **derived**, never written in. The specification forbids hard-coded
canonical counts and rubric E1's Fail descriptor names them.

> **→ YANDU.** The implicit precedence is **JSON-first**: `concat` puts the JSON frame first
> and `keep='first'` retains it. Not "no precedence rule" — "any deterministic choice is
> equivalent, and we chose JSON". Please record it that way.
>
> **Still waiting on you:** after this step the 500 shared orders are one row tagged
> `source_system = "JSON"`, so "appeared in both files" is gone before §5 sees the tables.
> Overlap key sets handed over pre-dedup, or a `"both"` marker?

`source_system` records provenance so §5 can report source coverage and
VAL-FLOW-12 can assert the overlap. It is a **helper column**: it never reaches
the exported CSVs, because `conform_to_contract` selects the dictionary's
fields rather than dropping by name, and the dictionary does not contain it.
VAL-SCHEMA-08 asserts its absence downstream.

**Order matters.** The marker is written *before* deduplication. `keep='first'`
retains the JSON copy of a shared key, whose `source_system` is `JSON`, so
marking afterwards would lose the very fact the column exists to record.

Marking first also keeps both copies of a duplicated key field-identical —
within-source pairs are both `JSON`, cross-source pairs are both `both` — so
the field-identity evidence behind DEC-017 still holds with the column present.

In [15]:
# --- §4.0.2 Pipeline helpers ---

BOTH_SOURCES = ['orders', 'order_items', 'deliveries', 'product_reviews']
JSON_ONLY    = ['customers']
XML_ONLY     = ['products']
NOT_AN_OUTPUT_TABLE = ['warehouses']


def combine_sources(table):
    """Normalise each source, then concatenate. Normalising first is required
    because `dayfirst` differs between the two files."""
    parts = []
    if table in json_tables:
        parts.append(normalise_frame(json_tables[table], table, dayfirst=False)
                     .assign(source_system='JSON'))
    if table in xml_tables and table not in NOT_AN_OUTPUT_TABLE:
        parts.append(normalise_frame(xml_tables[table], table, dayfirst=True)
                     .assign(source_system='XML'))
    return pd.concat(parts, ignore_index=True)

def mark_overlap(df, table):
    """Rewrite source_system to 'both' for keys carried by both sources.
    Must run before deduplication: keep='first' retains the JSON copy, so
    marking afterwards would lose the overlap fact."""
    key = CONTRACT[table]['pk']
    shared = df.groupby(key)['source_system'].transform('nunique') > 1
    out = df.copy()
    out.loc[shared, 'source_system'] = 'both'
    return out

def expected_canonical_rows(table):
    """Derived, never written in: the union of business keys across the sources."""
    key = CONTRACT[table]['pk']
    keys = set()
    for tabs in (json_tables, xml_tables):
        if table in tabs and table not in NOT_AN_OUTPUT_TABLE:
            keys |= set(tabs[table][key].astype(str))
    return len(keys)


def deduplicate(df, table):
    """One row per business key. Safe because within-source duplicates are
    field-identical (A3) and shared keys agree once normalised (A5)."""
    key = CONTRACT[table]['pk']
    out = df.drop_duplicates(subset=key, keep='first').reset_index(drop=True)

    assert len(out) == expected_canonical_rows(table), table
    assert out[key].is_unique
    assert out[key].notna().all() and (out[key].astype(str) != '').all()
    return out

In [16]:
# --- §4.0.2 Pipeline helpers, continued ---

def conform_to_contract(df, table):
    """Select the published fields in `position` order. Helper columns never survive
    this step — selection is what drops `source_system`."""
    fields = CONTRACT[table]['fields']
    missing = [f for f in fields if f not in df.columns]
    assert not missing, (table, 'missing fields', missing)

    out = df[fields].copy()
    assert list(out.columns) == fields          # B1 field order
    return out


def row_flow(table, combined, final):
    """The before/after evidence the template's §4 and rubric E1 both ask for."""
    per_source = {s: len(t[table]) for s, t in (('JSON', json_tables), ('XML', xml_tables))
                  if table in t}
    parts = ' + '.join(f'{s} {n:,}' for s, n in per_source.items())
    print(f'{table}')
    print(f'   sources   {parts} = {sum(per_source.values()):,} concatenated')
    print(f'   canonical {len(final):,} rows x {final.shape[1]} columns'
          f'   ({sum(per_source.values()) - len(final):,} removed)')

### 4.0.3 Money rounding and tolerance

Two things that look like detail and are not.

**`pandas .round(2)` and Python's `round()` disagree.** pandas rounds the scaled float
half-to-even; Python rounds the decimal value of the float. They part company on `.xx5`
boundaries — raw totals like `365.96500000000003`, `11687.645`, `2715.6749999999997`. Measured
on the JSON side: Python's `round()` reproduces the source exactly, 2,750 / 2,750; pandas
`.round(2)` reproduces 2,705 / 2,750, with the other 45 out by one cent.

The source was generated with Python's rounding, so §4 uses `money_round()` everywhere a
monetary value is rounded. Element-wise is acceptable at this scale — thousands of rows, not
the hundreds of thousands where the datetime parser needed vectorising.

**Tolerance means `<=`, not `<`.** The dictionary publishes `numeric tolerance 0.01` for
monetary fields, so a difference of exactly one cent is *inside* tolerance. Writing the check as
`gap > 0.01` reports 23 false failures, because float arithmetic makes a difference of exactly
0.01 evaluate as greater than 0.01. `np.isclose(a, b, rtol=0, atol=0.01)` is the correct form.

> **→ ALL.** Every monetary rounding in §4 goes through `money_round()`. If any of us rounds
> money with `.round(2)` the outputs will differ by a cent on a few dozen rows — inside
> tolerance, so it would not fail a check, but the six CSVs would stop being byte-reproducible
> between our runs.
>
> **→ YANDU.** Write the arithmetic checks as `|a − b| <= 0.01`, not `<`. Use
> `np.isclose(a, b, rtol=0, atol=0.01, equal_nan=True)` — `rtol=0` because the published rule is
> absolute, and `equal_nan=True` because `pd.NaT != pd.NaT` and `np.nan != np.nan` would
> otherwise report both-sides-missing as a conflict. A strict `>` gave 23 false failures on
> `order_total` before this was fixed.

In [17]:
# --- §4.0.3 Money rounding ---
# pandas .round(2) rounds the scaled float half-to-even; Python's round() rounds the
# decimal value of the float. They disagree on .xx5 boundaries, and the source was
# generated with Python's. Element-wise is fine at this scale (thousands of rows).

def money_round(series):
    return series.map(lambda v: round(v, 2))

TOLERANCE = 0.01
def within_tolerance(a, b, atol=TOLERANCE):
    """Tolerance means |a - b| <= atol. Float noise makes a strict > unreliable at the
    boundary, so np.isclose is used rather than a hand-written comparison."""
    import numpy as np
    return np.isclose(a, b, rtol=0, atol=atol, equal_nan=True)

### 4.2 `order_items`

Six fields, no naming exceptions, no WP4 dependency — built first because it is the cheapest
place to prove the pipeline before the harder tables use it.

`line_revenue` is the only derived field: `round(quantity × unit_price, 2)`.

The duplicate rows here are a *consequence* of the order duplication — 68 duplicated orders
carry their whole cart a second time — so the counts removed from `order_items`, `orders` and
`deliveries` move together. That relationship is a free consistency check.

In [18]:
# --- §4.2 order_items ---

TABLE = 'order_items'

combined = mark_overlap(combine_sources(TABLE), TABLE)
deduped  = deduplicate(combined, TABLE)

print(f'{TABLE}')
print(f'   JSON {len(json_tables[TABLE]):,} + XML {len(xml_tables[TABLE]):,}'
      f' = {len(combined):,} concatenated')
print(f'   deduped {len(deduped):,}  ({len(combined) - len(deduped):,} removed)')

deduped.head()

order_items
   JSON 8,826 + XML 8,833 = 17,659 concatenated
   deduped 15,685  (1,974 removed)


,source_system,line_revenue,order_id,order_item_id,product_id,quantity,unit_price
0,JSON,180.47,HORD000879,HITM0002731,PRD0045,1,180.47
1,JSON,1008.76,HORD000879,HITM0002732,PRD0183,1,1008.76
2,JSON,1338.24,HORD000537,HITM0001662,PRD0086,2,669.12
3,JSON,215.90,HORD000537,HITM0001663,PRD0640,1,215.90
4,JSON,2485.73,HORD000247,HITM0000766,PRD0921,1,2485.73


`line_revenue` is the only derived field in this table. The source carries a value for it, but
the pipeline **recomputes** it from `quantity` and `unit_price` rather than copying — the
specification prescribes the formula, and recomputing is what makes the arithmetic checkable.

Recomputing also gives a free reconciliation: the recomputed column should match the source
column within the published tolerance for every row. A mismatch would be a real finding.

> **→ YANDU.** `line_revenue`'s published `comparison_rule` is `numeric tolerance 0.01`. The
> check below uses that tolerance rather than exact equality, and reports the count rather than
> asserting it — a genuine mismatch should be visible, not fatal.

In [19]:
# --- §4.2 line_revenue: recompute and reconcile ---

recomputed = (deduped['quantity'] * deduped['unit_price']).round(2)
gap = (recomputed - deduped['line_revenue']).abs()

print(f'rows                     {len(deduped):,}')
print(f'outside tolerance 0.01   {int((gap > 0.01).sum()):,}')
print(f'largest difference       {gap.max():.4f}')

deduped['line_revenue'] = recomputed

rows                     15,685
outside tolerance 0.01   0
largest difference       0.0000


In [20]:
# --- §4.2 order_items: conform and report ---

order_items_marked = deduped
order_items_final  = conform_to_contract(order_items_marked, TABLE)
row_flow(TABLE, combined, order_items_final)

order_items_final.head()

order_items
   sources   JSON 8,826 + XML 8,833 = 17,659 concatenated
   canonical 15,685 rows x 6 columns   (1,974 removed)


,order_item_id,order_id,product_id,quantity,unit_price,line_revenue
0,HITM0002731,HORD000879,PRD0045,1,180.47,180.47
1,HITM0002732,HORD000879,PRD0183,1,1008.76,1008.76
2,HITM0001662,HORD000537,PRD0086,2,669.12,1338.24
3,HITM0001663,HORD000537,PRD0640,1,215.90,215.90
4,HITM0000766,HORD000247,PRD0921,1,2485.73,2485.73


### 4.4 `deliveries`

Second, because it exercises three things `order_items` did not: `date` fields, `boolean`
fields, and the grain filter.

**Grain: one row per completed order.** Strictly 1:1 with `orders` — after deduplication both
tables have the same row count and `order_id` is unique in each.

**The `completed` filter is a no-op on this package.** `order_status` and `delivery_status` are
100% `Completed` / `Delivered` across both files, so the filter removes zero rows. It is still
written, because the grain rule is part of the method and a reviewer needs to see it applied.

> **→ YANDU.** Two things here are trivially satisfied on this data and would make weak checks:
> the `completed` filter removes 0 rows, and any distribution check on `order_status` or
> `delivery_status` sees a single value. The 1:1 relationship with `orders` is the check worth
> writing.
>
> **→ YANDU.** `delay_reason == 'none'` is a real category on 4,472 rows, not a missing value.
> It must not be mapped to the `NaN` sentinel by any cleaning step.
`delay_days` is `max(0, delivered_date − promised_date)`, not the raw difference —
early arrivals record `0`, never a negative. Verified 5,000 of 5,000. Yandu first
wrote the check as the raw difference and got 2,985 false mismatches.
`delivered_date` extends past 31 December 2018 — 76 deliveries, latest
2019-01-12 — for orders placed late in the period. A temporal range check
therefore constrains `order_timestamp` only; written on `delivered_date` it
would report 76 false failures.

In [21]:
# --- §4.4 deliveries ---

TABLE = 'deliveries'

combined = mark_overlap(combine_sources(TABLE), TABLE)
deduped  = deduplicate(combined, TABLE)

deliveries_marked = deduped
deliveries_final  = conform_to_contract(deliveries_marked, TABLE)

row_flow(TABLE, combined, deliveries_final)

# 1:1 with orders — same count, and order_id unique in both.
print(f"\n   order_id unique      {deliveries_final['order_id'].is_unique}")
print(f"   delay_reason 'none'  {int((deliveries_final['delay_reason'] == 'none').sum()):,} rows")

# Not cleaned, and the evidence is that cleaning would damage it: the column
# holds two structured values, and clean_narrative_text alters all 5,000 rows
# by letter case alone. DEC-018.
notes = deliveries_final['delivery_note_clean']
print(f'delivery_note_clean: {notes.nunique()} distinct values')
print(notes.value_counts().to_string())

deliveries_final.head()

deliveries
   sources   JSON 2,818 + XML 2,818 = 5,636 concatenated
   canonical 5,000 rows x 20 columns   (636 removed)

   order_id unique      True
   delay_reason 'none'  4,472 rows
delivery_note_clean: 2 distinct values
delivery_note_clean
Delivered within promise    4472
Carrier scan reconciled      528


,delivery_id,order_id,dispatch_date,promised_date,delivered_date,carrier,service_level,delivery_status,delay_days,on_time_in_full,fulfilment_hours,delivery_cost,delay_reason,promised_days,tracking_event_count,delivery_window,shipping_distance_km,signature_required,estimated_carbon_kg,delivery_note_clean
0,HDEL000879,HORD000879,2019-01-01,2019-01-07,2019-01-08,DHL,Express,Delivered,1,False,24,19.13,warehouse_congestion,6,5,Afternoon,5.5105,False,0.421,Carrier scan reconciled
1,HDEL000537,HORD000537,2018-08-26,2018-09-02,2018-09-01,StarTrack,Express,Delivered,0,True,67,24.62,none,7,3,Anytime,5.9347,False,0.374,Delivered within promise
2,HDEL000247,HORD000247,2018-09-25,2018-09-29,2018-09-30,Direct Freight,Standard,Delivered,1,False,30,7.01,carrier_capacity,4,9,Afternoon,0.8042,True,0.927,Carrier scan reconciled
3,HDEL003196,HORD003196,2018-11-10,2018-11-15,2018-11-13,DHL,Standard,Delivered,0,True,13,8.53,none,5,9,Anytime,3.1114,True,1.238,Delivered within promise
4,HDEL003602,HORD003602,2018-12-03,2018-12-06,2018-12-04,StarTrack,Standard,Delivered,0,True,18,12.38,none,3,7,Morning,8.8545,True,0.574,Delivered within promise


### 4.3 `customers`

JSON only — no cross-source reconciliation. The deduplication step stays in the pipeline
anyway: its absence would be a gap in method, and its presence removing zero rows is evidence.

> **→ YANDU.** 497 of 500 customers have orders; three have none. That is not a foreign-key
> violation — the direction is `orders.customer_id → customers.customer_id`, which does not
> require every customer to appear. It does matter as a denominator in EDA.

In [22]:
# --- §4.3 customers ---

TABLE = 'customers'

combined = mark_overlap(combine_sources(TABLE), TABLE)
deduped  = deduplicate(combined, TABLE)
customers_marked = deduped
customers_final  = conform_to_contract(customers_marked, TABLE)

row_flow(TABLE, combined, customers_final)

# home_postcode is pure digits, so pandas would infer an integer on read-back.
# The six identifiers carry alpha prefixes and raise instead.
print(f"\n   home_postcode dtype  {customers_final['home_postcode'].dtype}")

customers_final.head()

customers
   sources   JSON 500 = 500 concatenated
   canonical 500 rows x 20 columns   (0 removed)

   home_postcode dtype  object


,customer_id,signup_date,loyalty_tier,customer_segment,age_band,preferred_channel,home_suburb,prior_12m_orders,lifetime_value_before_period,marketing_consent,home_postcode,home_state,home_country,preferred_language,acquisition_source,account_status,preferred_device,email_domain,household_size_band,contact_frequency_preference
0,CUS00001,2014-08-08,Bronze,Value,25-34,Web,Footscray,11,7283.87,True,3011,VIC,Australia,ja,Social,Active,Mobile,inbox.example,5+,Monthly
1,CUS00002,2013-09-21,Silver,Premium,25-34,Web,Southbank,10,6091.84,False,3006,VIC,Australia,ja,Referral,Active,Desktop,mail.test,5+,Quarterly
2,CUS00003,2013-07-21,Bronze,Mainstream,45-54,Store,Southbank,17,2231.02,True,3006,VIC,Australia,de,Store,Active,Mobile,inbox.example,3-4,Essential only
3,CUS00004,2013-07-11,Silver,Premium,55+,Web,Carlton,12,11466.57,True,3053,VIC,Australia,pl,Organic,Active,Desktop,example.net,1,Essential only
4,CUS00005,2014-05-03,Bronze,Value,45-54,Mobile,Brunswick,13,3488.02,True,3056,VIC,Australia,pl,Referral,Active,Tablet,example.net,5+,Monthly


### 4.5 `products`

XML only, and the most format-heavy table: every value arrives as text, so all eight typed
fields need converting — money with a conditional thousands comma, `DD/MM/YYYY` dates, `Y`/`N`
booleans.

This is the table WP1's hand-written `NORMALISERS` originally missed, because it appears in only
one source and so never showed up in the cross-source comparison the map was written for.

`product_description_clean` comes from `product_description_raw` via WP4. All 1,000 descriptions
carry a `[CATALOGUE]` marker and HTML tags, so this field stays provisional until G2.

In [23]:
# --- §4.5 products ---

TABLE = 'products'

combined = mark_overlap(combine_sources(TABLE), TABLE)
deduped  = deduplicate(combined, TABLE)

# Every typed field arrived as a string. Confirm the conversion actually happened.
print('dtypes after normalisation')
for kind, fields in PLAN[TABLE].items():
    for f in fields:
        print(f'   {kind:9s} {f:24s} {deduped[f].dtype}')

deduped.head()

dtypes after normalisation
   money     unit_price               float64
   money     unit_cost                float64
   date      launch_date              datetime64[ns]
   boolean   recyclable_packaging     bool
   boolean   active_flag              bool
   number    launch_year              int64
   number    warranty_months          int64
   number    weight_kg                float64


,source_system,product_id,product_name,category,brand,unit_price,unit_cost,launch_year,warranty_months,weight_kg,product_sku,subcategory,model_family,colour,supplier_id,supplier_country,launch_date,tax_category,package_type,recyclable_packaging,active_flag,product_description_raw
0,XML,PRD0001,Candle Bloom 100,Laptop,Candle,2765.47,1681.81,2017,24,1.209,SKU-CAN00001,Ultrabook,Arc,Green,SUP001,Australia,2017-07-01,GST_STANDARD,Recycled box,True,True,[CATALOGUE] <section>Ultrabook designed for po...
1,XML,PRD0002,Vela Halo 101,Smartphone,Vela,455.28,310.78,2014,24,0.280,SKU-VEL00002,5G Smartphone,Atlas,White,SUP002,Korea,2014-01-12,GST_STANDARD,Recycled box,True,True,[CATALOGUE] <section>5G Smartphone designed fo...
2,XML,PRD0003,Candle Quest 102,Tablet,Candle,472.24,295.51,2012,24,0.514,SKU-CAN00003,Productivity Tablet,Bloom,Black,SUP003,China,2012-02-01,GST_STANDARD,Recycled box,True,True,[CATALOGUE] <section>Productivity Tablet desig...
3,XML,PRD0004,Vela Wave 103,Home Entertainment,Vela,1300.01,720.70,2013,24,6.423,SKU-VEL00004,Smart Display,Core,Gold,SUP004,Malaysia,2013-11-05,GST_STANDARD,Recycled box,False,False,[CATALOGUE] <section>Smart Display designed fo...
4,XML,PRD0005,Candle Edge 104,Audio,Candle,770.31,459.45,2014,12,0.525,SKU-CAN00005,Wireless Audio,Edge,Black,SUP005,China,2014-11-04,GST_STANDARD,Protective case,False,True,[CATALOGUE] <section>Wireless Audio designed f...


In [24]:
# --- §4.5 products: derived field and conform ---

# product_description_clean comes from WP4. Provisional while the placeholder banner shows.
deduped['product_description_clean'] = deduped['product_description_raw'].map(clean_narrative_text)

products_marked = deduped
products_final  = conform_to_contract(products_marked, TABLE)
row_flow(TABLE, combined, products_final)

products_final.head()

products
   sources   XML 1,000 = 1,000 concatenated
   canonical 1,000 rows x 21 columns   (0 removed)


,product_id,product_name,category,brand,unit_price,unit_cost,launch_year,warranty_months,weight_kg,product_sku,subcategory,model_family,colour,supplier_id,supplier_country,launch_date,tax_category,package_type,recyclable_packaging,active_flag,product_description_clean
0,PRD0001,Candle Bloom 100,Laptop,Candle,2765.47,1681.81,2017,24,1.209,SKU-CAN00001,Ultrabook,Arc,Green,SUP001,Australia,2017-07-01,GST_STANDARD,Recycled box,True,True,ultrabook designed for portable document work ...
1,PRD0002,Vela Halo 101,Smartphone,Vela,455.28,310.78,2014,24,0.280,SKU-VEL00002,5G Smartphone,Atlas,White,SUP002,Korea,2014-01-12,GST_STANDARD,Recycled box,True,True,5g smartphone designed for daily communication...
2,PRD0003,Candle Quest 102,Tablet,Candle,472.24,295.51,2012,24,0.514,SKU-CAN00003,Productivity Tablet,Bloom,Black,SUP003,China,2012-02-01,GST_STANDARD,Recycled box,True,True,"productivity tablet designed for reading, anno..."
3,PRD0004,Vela Wave 103,Home Entertainment,Vela,1300.01,720.70,2013,24,6.423,SKU-VEL00004,Smart Display,Core,Gold,SUP004,Malaysia,2013-11-05,GST_STANDARD,Recycled box,False,False,smart display designed for shared film and tel...
4,PRD0005,Candle Edge 104,Audio,Candle,770.31,459.45,2014,12,0.525,SKU-CAN00005,Wireless Audio,Edge,Black,SUP005,China,2014-11-04,GST_STANDARD,Protective case,False,True,wireless audio designed for commuting and spok...


### 4.1 `orders`

The heaviest table. Three things happen here that happen nowhere else.

**The arithmetic chain, in this order.** Verified against all 2,818 rows of both sources.

```
line_revenue = round(quantity × unit_price, 2)          # §4.2, per item
order_price  = round(Σ line_revenue, 2)                 # summed from order_items
tax_amount   = round(order_price / 11, 2)               # BEFORE the discount
order_total  = round(order_price × (1 − coupon_discount/100) + delivery_charges, 2)
```

**A real finding, and what it was.** The first run of this section reported 40 rows where the
recomputed `order_total` differed from the source. `order_price` and `tax_amount` matched
exactly, which narrowed it to the last step. Two causes, both in the checking rather than the
data: `.round(2)` disagreeing with the source's rounding on `.xx5` boundaries, and a strict `>`
comparison rejecting differences of exactly one cent that the published tolerance permits. Both
are fixed in §4.0.3. No row of source data was wrong.

Worth recording rather than quietly correcting: a check that fires is only useful if what it
found is written down.


Three ways to get this wrong:

- **`coupon_discount` is a percentage, not a dollar amount.** Reading it as dollars reproduces
  only 640 of 2,818 order totals — and those 640 are exactly the rows where the discount is 0,
  so the error hides in plain sight.
- **`tax_amount` is computed before the discount and never added to `order_total`.**
- **Use Python's built-in `round()`.** It reproduces the source exactly, 2,818/2,818. `Decimal`
  with `ROUND_HALF_UP` matches 2,758 exactly and the other 60 differ by 0.01 — inside tolerance,
  so both are defensible, but one needs no explanation.

`order_price` is the only calculation that crosses tables. It does not change either table's
grain: `order_items` stays one row per item, `orders` stays one row per order.

> **→ YANDU.** Each of the four fields is reconciled against the source value at the published
> tolerance and the count is reported, not asserted — a genuine mismatch should be visible
> rather than fatal. `VAL-ARITH-` can reuse these comparisons directly.

In [25]:
# --- §4.1 orders: combine, deduplicate, rebuild the arithmetic ---

TABLE = 'orders'

combined = mark_overlap(combine_sources(TABLE), TABLE)
deduped  = deduplicate(combined, TABLE)

# order_price is the sum of the canonical line revenues, so it is rebuilt from §4.2's output
# rather than copied. Every order must have at least one item for this to be complete.
line_totals = order_items_final.groupby('order_id')['line_revenue'].sum().round(2)
assert deduped['order_id'].isin(line_totals.index).all(), 'orders with no items'

rebuilt = pd.DataFrame({'order_price': deduped['order_id'].map(line_totals)})
rebuilt['tax_amount']  = money_round(rebuilt['order_price'] / 11)
rebuilt['order_total'] = money_round(rebuilt['order_price'] * (1 - deduped['coupon_discount'] / 100)
                                     + deduped['delivery_charges'])

print(f'{TABLE}: recomputed vs source, tolerance {TOLERANCE}')
for f in ['order_price', 'tax_amount', 'order_total']:
    ok = within_tolerance(rebuilt[f], deduped[f])
    gap = (rebuilt[f] - deduped[f]).abs()
    print(f'   {f:14s} outside tolerance {int((~ok).sum()):>5,}   max diff {gap.max():.4f}')

for f in rebuilt.columns:
    deduped[f] = rebuilt[f]

orders: recomputed vs source, tolerance 0.01
   order_price    outside tolerance     0   max diff 0.0000
   tax_amount     outside tolerance     0   max diff 0.0000
   order_total    outside tolerance     0   max diff 0.0000


In [26]:
# --- §4.1 orders: derived text fields, sentinels, conform ---

# Two target fields come from one raw column via WP4. Provisional while the banner shows.
deduped['customer_note_clean'] = deduped['customer_note_raw'].map(clean_narrative_text)
deduped['promo_code']          = deduped['customer_note_raw'].map(extract_promo_code)

# The two nullable string fields in this table. The sentinel is the three characters 'NaN',
# so it is filled explicitly — .astype(str) would give lowercase 'nan' on pandas 2.x.
# coupon_code is a real source column - blanks become the literal sentinel.
cc = deduped['coupon_code']
deduped['coupon_code'] = cc.where(cc.notna() & (cc != ''), 'NaN')



orders_marked = deduped
orders_final  = conform_to_contract(orders_marked, TABLE)
row_flow(TABLE, combined, orders_final)

print(f"\n   coupon_code == 'NaN'  {int((orders_final['coupon_code'] == 'NaN').sum()):,} rows")
print(f"   promo_code  == 'NaN'  {int((orders_final['promo_code']  == 'NaN').sum()):,} rows")

orders_final.head()

orders
   sources   JSON 2,818 + XML 2,818 = 5,636 concatenated
   canonical 5,000 rows x 23 columns   (636 removed)

   coupon_code == 'NaN'  3,127 rows
   promo_code  == 'NaN'  3,127 rows


,order_id,source_system_record_id,customer_id,order_timestamp,sales_channel,payment_method,currency,nearest_warehouse,order_status,order_price,delivery_charges,coupon_code,coupon_discount,tax_amount,order_total,season,expedited_delivery,customer_lat,customer_long,device_type,referral_source,customer_note_clean,promo_code
0,HORD000879,SRC-001-H-000879,CUS00481,2018-12-31 13:52:00,Web,Gift Card,AUD,Bakers,Completed,1189.23,21.86,NaN,10.0,108.11,1092.17,Summer,True,-37.853022,145.026366,Mobile,Paid Search,gift purchase for a family member,NaN
1,HORD000537,SRC-001-H-000537,CUS00242,2018-08-23 14:56:00,Web,Bank Transfer,AUD,Nickolson,Completed,1554.14,28.04,B1SAVE-66,25.0,141.29,1193.64,Winter,True,-37.871966,144.970039,Tablet,Organic,gift purchase for a family member,B1SAVE-66
2,HORD000247,SRC-001-H-000247,CUS00321,2018-09-24 08:07:00,Web,Bank Transfer,AUD,Bakers,Completed,4732.56,8.09,NaN,10.0,430.23,4267.39,Spring,False,-37.802892,144.993514,Tablet,Paid Search,please use minimal packaging,NaN
3,HORD003196,SRC-001-H-003196,CUS00474,2018-11-09 17:37:00,Mobile,Card,AUD,Thompson,Completed,3736.46,9.77,NaN,5.0,339.68,3559.41,Spring,False,-37.840529,144.943718,Tablet,Referral,no special instruction,NaN
4,HORD003602,SRC-001-H-003602,CUS00221,2018-12-02 08:33:00,Mobile,Bank Transfer,AUD,Bakers,Completed,4170.10,14.20,B1SAVE-11,20.0,379.10,3350.28,Summer,False,-37.876453,145.050783,Tablet,Referral,no special instruction,B1SAVE-11


### 4.6 `product_reviews`

Built last: seven of its 21 fields are derived and every one depends on WP4.

**The derivation order matters.** Three of the seven come from the raw text, four come from the
*cleaned* text — the specification is explicit that the Latin analysis and the counts are built
from `review_body_clean`, not from the raw value.

| target field | derived from |
|---|---|
| `review_body_clean` | `review_body_raw` |
| `extracted_order_reference`, `extracted_product_sku` | `review_body_raw` |
| `review_body_latin_analysis` | `review_body_clean` |
| `review_length_chars`, `review_word_count` | `review_body_clean` |
| `contains_non_latin_script` | `review_body_clean` |

Four foreign keys — `order_id`, `order_item_id`, `product_id`, `customer_id` — more than any
other table. `order_item_id` is unique across the canonical reviews, so the relationship to
`order_items` is 1:1 and no order item is reviewed twice. Joining the two cannot multiply rows.

`review_title` is a direct copy. WP1 checked all 7,892 and none contain markup, markers, URLs,
entities or uppercase — that count is the evidence for not cleaning it.

> **→ SHAWN.** Two of these read from your *output*, not from the raw text:
> `build_latin_analysis` and `contains_non_latin_script` are called on `review_body_clean`. If
> your implementation expects the raw value instead, say so now — the results would differ
> silently rather than raise.
>
> **→ SHAWN + YANDU.** The specification says the length and word counts must preserve the
> sentinel rather than counting the three characters of `'NaN'`. Both fields are
> `nullable = False` and typed `number`, so what they should hold when the clean text is the
> sentinel is unresolved. It cannot be tested until the real functions land; if no review ever
> cleans to empty the question disappears.

**Sentinel handling in the two review measures.** When `review_body_clean` is the
literal `NaN`, `str.len()` returns 3 and `str.split()` returns 1 — the three
letters counted as an ordinary review, which the specification forbids.

No row in this package is affected: nothing cleans to the sentinel. The guard is
here for a private test case or a re-run on different data, where a silent 3 and
1 would be indistinguishable from a real short review.

The value is derived, not chosen: the dictionary types both fields as `number`
with `nullable = False`, so the sentinel cannot propagate into them, and the
length of absent text is 0.

**Shawn** — both fields are yours in the mapping. Worth one clause in each row
saying the measures are taken on the cleaned body and that the sentinel maps to 0.

In [27]:
# --- §4.6 product_reviews ---

TABLE = 'product_reviews'

combined = mark_overlap(combine_sources(TABLE), TABLE)
deduped  = deduplicate(combined, TABLE)

# From the raw text.
deduped['review_body_clean']         = deduped['review_body_raw'].map(clean_narrative_text)
deduped['extracted_order_reference'] = deduped['review_body_raw'].map(extract_order_reference)
deduped['extracted_product_sku']     = deduped['review_body_raw'].map(extract_product_sku)

# From the cleaned text, not the raw value.
clean = deduped['review_body_clean']
deduped['review_body_latin_analysis'] = clean.map(build_latin_analysis)
deduped['contains_non_latin_script']  = clean.map(contains_non_latin_script)

# The sentinel is not an ordinary review: str.len() would return 3 and
# str.split() 1. Both fields are typed number / nullable False, so 0.
is_sentinel = clean.eq('NaN')
deduped['review_length_chars']       = clean.str.len().mask(is_sentinel, 0)
deduped['review_word_count']         = clean.str.split().str.len().mask(is_sentinel, 0)

product_reviews_marked = deduped
product_reviews_final  = conform_to_contract(product_reviews_marked, TABLE)
row_flow(TABLE, combined, product_reviews_final)

product_reviews_final.head()

product_reviews
   sources   JSON 3,946 + XML 3,946 = 7,892 concatenated
   canonical 7,000 rows x 21 columns   (892 removed)


,review_id,order_id,order_item_id,product_id,customer_id,review_timestamp,language_code,rating,review_title,review_body_clean,review_body_latin_analysis,verified_purchase,helpful_votes,review_length_chars,review_word_count,contains_non_latin_script,extracted_order_reference,extracted_product_sku,delivery_experience,value_experience,writing_style
0,HREV002000,HORD001451,HITM0004524,PRD0108,CUS00500,2018-05-23 10:20:00,en,5,useful daily tracking,vela spark 207 has become part of my morning a...,vela spark 207 has become part of my morning a...,True,33,982,161,False,HORD001451,SKU-VEL00108,on_time,good_value,detailed
1,HREV002727,HORD001981,HITM0006194,PRD0173,CUS00186,2018-06-16 17:27:00,en,3,useful for forms and visits,"i have used candle vista 272 for field forms, ...","i have used candle vista 272 for field forms, ...",True,61,457,73,False,HORD001981,SKU-CAN00173,on_time,good_value,concise
2,HREV003841,HORD002749,HITM0008656,PRD0710,CUS00107,2018-03-19 11:01:00,en,5,flexible mounting for a crowded desk,vela atlas 809 has made a bigger difference to...,vela atlas 809 has made a bigger difference to...,True,84,1435,241,False,HORD002749,SKU-VEL00710,on_time,good_value,comparison
3,HREV003458,HORD002475,HITM0007790,PRD0363,CUS00166,2018-10-29 18:38:00,en,4,a flexible companion for media and video calls,candle quest 462 has worked better for portabl...,candle quest 462 has worked better for portabl...,True,82,1503,248,False,HORD002475,SKU-CAN00363,on_time,good_value,comparison
4,HREV006699,HORD004771,HITM0015004,PRD0477,CUS00344,2018-11-10 19:39:00,en,4,clear communication during competitive play,i chose the candle flux 576 for evenings of co...,i chose the candle flux 576 for evenings of co...,True,60,1834,305,False,HORD004771,SKU-CAN00477,on_time,poor_value,narrative


In [28]:
# Pre-deduplication frames, kept for WP3's §5 reconciliation (VAL-FLOW-09/10/12).
COMBINED = {t: mark_overlap(combine_sources(t), t) for t in OUTPUT_TABLES}
print({t: len(df) for t, df in COMBINED.items()})

{'orders': 5636, 'order_items': 17659, 'customers': 500, 'deliveries': 5636, 'products': 1000, 'product_reviews': 7892}


Three target fields cannot be copied from either source — `orders.promo_code`,
`product_reviews.extracted_order_reference`, `product_reviews.extracted_product_sku`.
They are extracted from narrative text by WP4.

While §3 falls back to placeholders, those three columns are legitimately all
sentinel and nothing else may be. Once the real text functions load, none of
them is: the expected set becomes empty. The assertion below switches on
`WP4_PLACEHOLDER` so it stays meaningful in both states rather than needing a
hand edit when WP4 lands — which is exactly the edit someone forgets.

This mirrors WP3's VAL-FLOW-13. If a column outside the expected set is ever
all sentinel, a real source column was dropped upstream and the export would
ship a dead column silently.

In [29]:
# Before WP4 lands these three are legitimately all-sentinel; afterwards none is.
PENDING_WP4 = {('orders', 'promo_code'),
               ('product_reviews', 'extracted_order_reference'),
               ('product_reviews', 'extracted_product_sku')}
expected = PENDING_WP4 if WP4_PLACEHOLDER else set()

dead = {(name, c)
        for name in OUTPUT_TABLES
        for df in [globals()[f'{name}_final']]
        for c in df.columns if (df[c].astype(str) == 'NaN').all()}

assert dead == expected, f'unexpected all-sentinel columns: {dead ^ expected}'
print(f'{len(dead)} all-sentinel columns (WP4 placeholders in use: {WP4_PLACEHOLDER})')

0 all-sentinel columns (WP4 placeholders in use: False)


---

## 7. Export the six CSV files

B1 is assessed on the exported artefacts: exact filenames, exact columns, exact order. Everything
upstream can be right and still lose the mark here.

**Temporal values are formatted only at this point.** They stayed as Timestamps through §4 so
that comparison and deduplication worked on typed values. The dictionary distinguishes `date`
from `datetime`, and the two take different string forms — five fields are `date`
(`signup_date`, `dispatch_date`, `promised_date`, `delivered_date`, `launch_date`, across three
tables) and two are `datetime` (`order_timestamp`, `review_timestamp`).

**Written to a personal folder.** `OUTPUT_DIR` is scratch in this notebook (D9). Only a full
clean run of `00_Master/Group001_solution.ipynb` writes to the shared `02_Outputs/`.

**Read back before believing it.** Writing is not the same as writing correctly. Each file is
re-read with `keep_default_na=False` and `dtype=str` so the literal `NaN` sentinel is visible as
three characters rather than silently reinterpreted as a float NaN, and so identifier padding
survives the round trip.

> **→ YANDU.** The read-back block is the natural home for `VAL-SCHEMA-` — it already checks
> field presence, field order and primary-key uniqueness against the dictionary rather than
> against a hard-coded list.

In [30]:
# --- §7 Format temporal values and export ---

# DEC-022: output built on placeholder text functions never lands beside the real
# thing. The banner in §3 scrolls away; a different folder does not.
if WP4_PLACEHOLDER:
    OUTPUT_DIR = OUTPUT_DIR / 'provisional'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def format_for_export(df, table):
    """Timestamps -> the published string forms. `date` and `datetime` differ."""
    out = df.copy()
    plan = PLAN.get(table, {})
    for f in plan.get('date', []):
        out[f] = out[f].dt.strftime('%Y-%m-%d')
    for f in plan.get('datetime', []):
        out[f] = out[f].dt.strftime('%Y-%m-%d %H:%M:%S')
    return out


TABLES = {
    'orders':          orders_final,
    'order_items':     order_items_final,
    'customers':       customers_final,
    'deliveries':      deliveries_final,
    'products':        products_final,
    'product_reviews': product_reviews_final,
}

for name, df in TABLES.items():
    path = OUTPUT_DIR / f'{GROUP_ID}_{name}_standardised.csv'
    format_for_export(df, name).to_csv(path, index=False)

    # Read back and verify the contract survived the round trip.
    back = pd.read_csv(path, keep_default_na=False, dtype=str)
    fields = CONTRACT[name]['fields']
    pk = CONTRACT[name]['pk']
    assert list(back.columns) == fields, name
    assert len(back) == len(df), name
    assert back[pk].is_unique and (back[pk] != '').all(), name

    print(f'{name:16s} {len(back):>7,} rows x {back.shape[1]:>2} cols  ->  {path.name}')

orders             5,000 rows x 23 cols  ->  Group001_orders_standardised.csv
order_items       15,685 rows x  6 cols  ->  Group001_order_items_standardised.csv
customers            500 rows x 20 cols  ->  Group001_customers_standardised.csv
deliveries         5,000 rows x 20 cols  ->  Group001_deliveries_standardised.csv
products           1,000 rows x 21 cols  ->  Group001_products_standardised.csv
product_reviews    7,000 rows x 21 cols  ->  Group001_product_reviews_standardised.csv


---

## 2. Source-to-target mapping — WP2's rows

> **WP1 — Echo curates.** She owns the file and its phrasing (D6). This section produces only
> the `transformation_or_derivation` and `notebook_evidence` entries for the rows my code
> produces, as a two-column handover keyed on `output_table` + `target_field`.

101 of the 111 rows are mine — every field except the ten derived by WP4. Most are formulaic:
the transformation follows from the normalisation category, which was itself derived from the
dictionary in §4.0. Writing them by hand would be 101 chances to describe something the code
does not do.

Six rows are not formulaic and are written individually: the four arithmetic fields, and the two
where a sentinel is filled.

In [31]:
# --- §2 WP2's mapping rows ---

WP4_DERIVED = {
    ('orders', 'customer_note_clean'), ('orders', 'promo_code'),
    ('products', 'product_description_clean'),
    ('product_reviews', 'review_body_clean'),
    ('product_reviews', 'review_body_latin_analysis'),
    ('product_reviews', 'review_length_chars'),
    ('product_reviews', 'review_word_count'),
    ('product_reviews', 'contains_non_latin_script'),
    ('product_reviews', 'extracted_order_reference'),
    ('product_reviews', 'extracted_product_sku'),
}

BY_CATEGORY = {
    'money':    "Strip the 'AUD' label and thousands separator, cast to float, round to 2dp "
                "(§4.0.1 norm_money, §4.0.3 money_round). JSON arrives numeric; one function "
                "accepts both spellings.",
    'percent':  "Strip '%' and cast to float. Percentage points, not a fraction — arithmetic "
                "divides by 100 (§4.0.1 norm_percent).",
    'boolean':  "XML 'Y'/'N' and JSON native bool both become Python bool (§4.0.1 norm_bool).",
    'date':     "Parsed dayfirst for XML and ISO for JSON, held as a Timestamp through §4, "
                "emitted as YYYY-MM-DD at export (§4.0.1, §7).",
    'datetime': "Parsed dayfirst for XML and ISO for JSON, held as a Timestamp through §4, "
                "emitted as YYYY-MM-DD HH:MM:SS at export (§4.0.1, §7).",
    'number':   "Cast to numeric (§4.0.1).",
}
DEFAULT = ("Direct copy after deduplication. No case change; identifier padding preserved "
           "(dtype held as string through export).")

INDIVIDUAL = {
    ('order_items', 'line_revenue'):
        "Recomputed as round(quantity × unit_price, 2) from the normalised columns; the source "
        "value is not copied. Reconciled against the source at the published tolerance 0.01 — "
        "0 of 15,685 rows outside, max difference 0.0000 (§4.2).",
    ('orders', 'order_price'):
        "Recomputed as round(Σ line_revenue, 2) over the canonical order_items of each order; "
        "the source value is not copied. Reconciled at tolerance 0.01 — 0 rows outside (§4.1).",
    ('orders', 'tax_amount'):
        "Recomputed as round(order_price / 11, 2), before the coupon discount, and reported "
        "separately rather than added to order_total. Reconciled at 0.01 — 0 rows outside (§4.1).",
    ('orders', 'order_total'):
        "Recomputed as round(order_price × (1 − coupon_discount/100) + delivery_charges, 2); "
        "coupon_discount is percentage points. Reconciled at 0.01 — 0 rows outside (§4.1).",
    ('orders', 'coupon_code'):
        "Direct copy. Missing is an empty string in the JSON and an empty element in the XML; "
        "both become the literal three-character 'NaN' sentinel, filled before any string cast "
        "(§4.1).",
    ('deliveries', 'delivery_note_clean'):
        "Direct copy from the source column of the same name. Deliberately not "
        "cleaned: the field holds two structured values and clean_narrative_text "
        "alters all 5,000 rows by letter case alone, which would corrupt a "
        "categorical (DEC-018).",
}

SECTION = {'orders': '§4.1', 'order_items': '§4.2', 'customers': '§4.3',
           'deliveries': '§4.4', 'products': '§4.5', 'product_reviews': '§4.6'}

def category_of(table, field):
    for kind, fields in PLAN.get(table, {}).items():
        if field in fields:
            return kind
    return None

rows = []
for _, f in dd.iterrows():
    key = (f.output_table, f.field_name)
    if key in WP4_DERIVED:
        text = 'TODO-SHAWN'
    elif key in INDIVIDUAL:
        text = INDIVIDUAL[key]
    else:
        text = BY_CATEGORY.get(category_of(*key), DEFAULT)
    rows.append({'output_table': f.output_table, 'target_field': f.field_name,
                 'transformation_or_derivation': text,
                 'notebook_evidence': SECTION[f.output_table]})

wp2_mapping = pd.DataFrame(rows)
wp2_mapping.to_csv(OUTPUT_DIR / f'{GROUP_ID}_mapping_wp2_rows.csv', index=False)

print(wp2_mapping.transformation_or_derivation.eq('TODO-SHAWN').value_counts().to_string())
print()
print(wp2_mapping.loc[wp2_mapping.output_table == 'order_items'].to_string(index=False))

transformation_or_derivation
False    101
True      10

output_table  target_field                                                                                                                                                                                                       transformation_or_derivation notebook_evidence
 order_items order_item_id                                                                                                               Direct copy after deduplication. No case change; identifier padding preserved (dtype held as string through export).              §4.2
 order_items      order_id                                                                                                               Direct copy after deduplication. No case change; identifier padding preserved (dtype held as string through export).              §4.2
 order_items    product_id                                                                                                      

#### Handover to WP1 (Echo)

This cell writes WP2's share of the source-to-target mapping to
`outputs_wip_jasmine/Group001_mapping_wp2_rows.csv`.

**Echo** — these are the 101 rows your §2 skeleton marks `TODO-JASMINE`.
Two columns are filled: `transformation_or_derivation` and `notebook_evidence`.
Everything else (`output_table`, `target_field`, `source_file`, `source_path`,
`source_format`, `data_type`, `nullable`) comes from your skeleton unchanged —
I did not touch the key columns, so a straight merge on
`(output_table, target_field)` is safe.

The `notebook_evidence` values cite section numbers in **this** notebook
(§4.0.1, §4.1 …). Once WP2's cells are merged into the final
`Group001_A1_solution.ipynb`, the numbering carries over unchanged — that is
what D4 (copy the template's numbering verbatim) buys us, so no re-pointing
is needed.

**Shawn** — the 10 rows printed below are left blank on purpose. They are the
text-derived fields owned by WP4; the transformation text has to describe your
functions, not my placeholder copy. Please fill them in the same two columns
and send them to Echo.

In [32]:
# Which (table, field) pairs does the dictionary require that WP2 did not produce?
required = set(zip(dd['output_table'], dd['field_name']))
covered  = set(zip(wp2_mapping['output_table'], wp2_mapping['target_field']))
print('dictionary rows:', len(required), '| produced by WP2:', len(covered))

missing = sorted(required - covered)
print(f'\nleft for WP4 ({len(missing)}):')
for t, f in missing:
    print(f'  {t:<16} {f}')

dictionary rows: 111 | produced by WP2: 111

left for WP4 (0):


In [33]:
# WP4's rows are not blank - they carry a TODO placeholder instead.
tod = wp2_mapping['transformation_or_derivation']
is_wp4 = tod.str.contains('TODO', case=False, na=False)

print(f'{(~is_wp4).sum()} written by WP2, {is_wp4.sum()} placeholders for WP4:\n')
print(wp2_mapping.loc[is_wp4, ['output_table', 'target_field']].to_string(index=False))

assert is_wp4.sum() == len(WP4_DERIVED), f'expected {len(WP4_DERIVED)} WP4 placeholders, got {is_wp4.sum()}'
assert wp2_mapping.loc[~is_wp4, 'notebook_evidence'].str.startswith('§').all(), \
    'a WP2 row cites no notebook section'

out_path = OUTPUT_DIR / f'{GROUP_ID}_mapping_wp2_rows.csv'
wp2_mapping.to_csv(out_path, index=False)
print(f'\n{len(wp2_mapping)} rows -> {out_path}')

101 written by WP2, 10 placeholders for WP4:

   output_table               target_field
         orders        customer_note_clean
         orders                 promo_code
       products  product_description_clean
product_reviews          review_body_clean
product_reviews review_body_latin_analysis
product_reviews        review_length_chars
product_reviews          review_word_count
product_reviews  contains_non_latin_script
product_reviews  extracted_order_reference
product_reviews      extracted_product_sku

111 rows -> outputs_wip_jasmine/Group001_mapping_wp2_rows.csv


In [34]:
# Any field whose name implies text derivation - which side did it land on?
PAT = r'_clean$|^extracted_|^contains_|_analysis$|_chars$|_count$'
suspect = dd.loc[dd['field_name'].str.contains(PAT, regex=True),
                 ['output_table', 'field_name']]

owner = wp2_mapping.set_index(['output_table', 'target_field'])['transformation_or_derivation']
for t, f in suspect.itertuples(index=False):
    side = 'WP4' if 'TODO' in str(owner.loc[(t, f)]) else 'WP2'
    print(f'{side:<4} {t:<16} {f}')

# Does promo_code exist as a source column at all, or must it be extracted?
print('\npromo_code in orders_final:', 'promo_code' in orders_final.columns)
print(orders_final['promo_code'].head(5).tolist() if 'promo_code' in orders_final.columns else '')

WP4  orders           customer_note_clean
WP2  deliveries       tracking_event_count
WP2  deliveries       delivery_note_clean
WP4  products         product_description_clean
WP4  product_reviews  review_body_clean
WP4  product_reviews  review_body_latin_analysis
WP4  product_reviews  review_length_chars
WP4  product_reviews  review_word_count
WP4  product_reviews  contains_non_latin_script
WP4  product_reviews  extracted_order_reference
WP4  product_reviews  extracted_product_sku

promo_code in orders_final: True
['NaN', 'B1SAVE-66', 'NaN', 'NaN', 'B1SAVE-11']


In [35]:
CHECK = ['promo_code', 'coupon_code', 'delivery_note_clean', 'tracking_event_count']
cols = ['output_table', 'target_field', 'source_file', 'source_path', 'source_format']
print(wp2_mapping.loc[wp2_mapping['target_field'].isin(CHECK),
                      [c for c in cols if c in wp2_mapping.columns]].to_string(index=False))

# How much of promo_code is real content versus sentinel?
pc = orders_final['promo_code']
print(f'\npromo_code: {(pc != "NaN").sum()} real values of {len(pc)}')
print(pc[pc != 'NaN'].head().tolist())

output_table         target_field
      orders          coupon_code
      orders           promo_code
  deliveries tracking_event_count
  deliveries  delivery_note_clean

promo_code: 1873 real values of 5000
['B1SAVE-66', 'B1SAVE-11', 'B1SAVE-66', 'B1SAVE-17', 'B1SAVE-68']


In [36]:
# Which parsed frames still in memory carry a promo or note column?
for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        hits = [c for c in obj.columns if 'promo' in c.lower() or 'note' in c.lower()]
        if hits:
            print(f'{name:<26} {len(obj):>6} rows   {hits}')

__                              5 rows   ['customer_note_clean', 'promo_code']
deliveries_marked            5000 rows   ['delivery_note_clean']
deliveries_final             5000 rows   ['delivery_note_clean']
_21                             5 rows   ['delivery_note_clean']
orders_marked                5000 rows   ['customer_note_raw', 'customer_note_clean', 'promo_code']
orders_final                 5000 rows   ['customer_note_clean', 'promo_code']
_26                             5 rows   ['customer_note_clean', 'promo_code']
wp2_mapping                   111 rows   ['notebook_evidence']


In [37]:
# Diagnostic only: an existence count on the raw bytes, lower-cased so the
# search is not defeated by the source's own casing. This is NOT structural
# parsing - it never decides field boundaries, it only answers "does this key
# appear in the source at all".
for path in (JSON_PATH, XML_PATH):
    blob = path.read_bytes().lower()
    print(path.name)
    for key in (b'promo', b'promo_code', b'promocode', b'coupon_code', b'customer_note'):
        print(f'   {key.decode():<16} {blob.count(key):>8}')

Group001_commerce.json
   promo                1051
   promo_code              0
   promocode               0
   coupon_code             0
   customer_note           0
Group001_operations.xml
   promo                1048
   promo_code              0
   promocode               0
   coupon_code          3866
   customer_note        5636


In [38]:
for name in ['orders_final', 'order_items_final', 'customers_final',
             'deliveries_final', 'products_final', 'product_reviews_final']:
    df = globals()[name]
    dead = [c for c in df.columns if (df[c].astype(str) == 'NaN').all()]
    print(f'{name:<22} all-NaN columns: {dead if dead else "none"}')

orders_final           all-NaN columns: none
order_items_final      all-NaN columns: none
customers_final        all-NaN columns: none
deliveries_final       all-NaN columns: none
products_final         all-NaN columns: none
product_reviews_final  all-NaN columns: none


In [39]:
# --- B1 evidence: shape, key integrity and grain, per table ---
rows = []
for t in OUTPUT_TABLES:
    df, c = globals()[f'{t}_final'], CONTRACT[t]
    pk = c['pk']
    rows.append({'table': t, 'rows': len(df), 'fields': len(c['fields']),
                 'pk_unique': df[pk].is_unique,
                 'pk_complete': (df[pk].astype(str).str.strip() != '').all(),
                 'grain_holds': len(df) == df[pk].nunique()})

b = pd.DataFrame(rows)
print(b.to_string(index=False))

assert b[['pk_unique', 'pk_complete', 'grain_holds']].all().all(), \
    b.loc[~b[['pk_unique', 'pk_complete', 'grain_holds']].all(axis=1), 'table'].tolist()

          table  rows  fields  pk_unique  pk_complete  grain_holds
         orders  5000      23       True         True         True
    order_items 15685       6       True         True         True
      customers   500      20       True         True         True
     deliveries  5000      20       True         True         True
       products  1000      21       True         True         True
product_reviews  7000      21       True         True         True


**Boolean encoding differs between the two sources.** The JSON writes `True`,
the XML writes `Y`. Both normalise to Python `True` here.

This is why the pipeline order matters: concatenate → normalise → deduplicate.
Comparing the two sources *before* normalisation would report all 3,946 shared
review rows as conflicting on `verified_purchase`, which is an artefact of
encoding, not a real disagreement.

**Yandu** — `product_reviews.verified_purchase` is single-valued: `True` on all
7,000 canonical rows, and single-valued in both sources independently. An
allowed-value check will pass trivially; a "both values present" check would
raise a false FAIL. Same situation as `order_status` and `delivery_status`.

**Whoever takes the review EDA figure** — this column cannot carry a
visualisation. Zero variance.

In [40]:
# --- B3 self-check: boolean encoding, and sentinels in non-nullable fields ---
FALSEY = {'false', 'no', '0', 'n'}

for t in OUTPUT_TABLES:
    df, c = globals()[f'{t}_final'], CONTRACT[t]
    for f in c['fields']:
        if str(c['dtype'][f]).strip().lower() in ('boolean', 'bool'):
            print(f"bool   {t:<16} {f:<28} {sorted(df[f].astype(str).unique())[:6]}")

print()
for t in OUTPUT_TABLES:
    df, c = globals()[f'{t}_final'], CONTRACT[t]
    for f in c['fields']:
        if str(c['nullable'][f]).strip().lower() in FALSEY:
            n = int((df[f].astype(str) == 'NaN').sum())
            if n:
                print(f"sentinel in nullable=False   {t:<16} {f:<28} {n:>6} rows")

bool   orders           expedited_delivery           ['False', 'True']
bool   customers        marketing_consent            ['False', 'True']
bool   deliveries       on_time_in_full              ['False', 'True']
bool   deliveries       signature_required           ['False', 'True']
bool   products         recyclable_packaging         ['False', 'True']
bool   products         active_flag                  ['False', 'True']
bool   product_reviews  verified_purchase            ['True']
bool   product_reviews  contains_non_latin_script    ['False', 'True']



In [41]:
print('combined columns:', 'verified_purchase' in combined.columns, '|', len(combined), 'rows')
if 'verified_purchase' in combined.columns:
    print(combined['verified_purchase'].astype(str).value_counts(dropna=False).head(10))

combined columns: True | 7892 rows
verified_purchase
True    7892
Name: count, dtype: int64


**Cross-validation of WP4's `extract_promo_code`, by an independent route.**

`coupon_code` is a structured source column read by the parser. `promo_code` is
extracted by regular expression from the free-text customer note. Nothing in
the pipeline derives one from the other.

On the canonical 5,000 orders they agree completely: 1,873 rows carry both and
disagree on none, 3,127 rows carry the sentinel in both, and **no row has one
populated while the other is not**. That last figure is the strong one — it
rules out missed extractions and spurious ones at the same time, which a
pattern check on `promo_code` alone cannot do.

This is WP3's VAL-TEXT-13. It is the only check in the register that can
distinguish a correctly-shaped wrong answer from a correct one, because every
other text check tests WP4's output against WP4's own pattern.

**Shawn** — this is evidence your extractor is right, not just well-formed.
**Yandu** — the baseline is no longer vacuous: 1,873 comparable rows, 0 defects.

In [42]:
# VAL-TEXT-13 evidence: two independent routes to the same fact.
cc, pc = orders_final['coupon_code'], orders_final['promo_code']

print('rows where the populated/sentinel pattern differs:', int((cc == 'NaN').ne(pc == 'NaN').sum()))

both = orders_final.loc[(cc != 'NaN') & (pc != 'NaN')]
disagree = both.loc[both['coupon_code'] != both['promo_code']]
print(f'{len(both):,} rows carry both, {len(disagree):,} disagree')
print(disagree[['order_id', 'coupon_code', 'promo_code']].head(10).to_string(index=False))

rows where the populated/sentinel pattern differs: 0
1,873 rows carry both, 0 disagree
Empty DataFrame
Columns: [order_id, coupon_code, promo_code]
Index: []


In [43]:
c = CONTRACT['product_reviews']
for f in ['review_length_chars', 'review_word_count']:
    print(f, '|', c['dtype'][f], '| nullable:', c['nullable'][f], '| rule:', c['comparison'][f])

review_length_chars | number | nullable: False | rule: exact after published normalisation
review_word_count | number | nullable: False | rule: exact after published normalisation


In [44]:
# The byte counts above are a smell test. The real number comes from the frame.
m = orders_marked
pop = m['coupon_code'].notna() & (m['coupon_code'] != '') & (m['coupon_code'] != 'NaN')
print(m.loc[pop, 'source_system'].value_counts().to_string())
print(f"\ncanonical orders with a coupon code: {int(pop.sum()):,} of {len(m):,}")

source_system
XML     843
JSON    842
both    188

canonical orders with a coupon code: 1,873 of 5,000


**Why the counts differ between the two files, and what they prove.**

The search is case-folded, so a zero here means the key is genuinely absent, not
merely spelled differently — except across the two files, which use different
conventions: the JSON writes camelCase (`couponCode`, `customerNote`) and the
XML writes snake_case (`Coupon_Code`, `Customer_Note`). Both spellings are
searched.

`promo_code` and `promocode` are 0 in both files. The target field does not
exist in either source and can only be extracted from narrative text (DEC-018
neighbourhood; see §4.1).

The XML count of 3,866 for `coupon_code` is not 2 × 1,933. Empty elements are
self-closing and contribute one occurrence; populated ones contribute an opening
and a closing tag. With 2,818 order records, `2p + (2818 − p) = 3866` gives
**p = 1,048** populated elements — which is exactly the XML count for `promo`.

The parsed frames agree: pre-deduplication, 1,051 JSON records and 1,048 XML
records carry a populated `coupon_code`. So every order record carrying a coupon
code also carries the string `promo` in its note text, independently in both
files, before any parsing decision was made.

This is a byte-level existence count, not structural parsing. It decides nothing
and produces no value; it is reported because it corroborates VAL-TEXT-13 from a
completely different direction.

In [45]:
# Pre-dedup, per file - directly comparable to the byte counts above.
raw = combine_sources('orders')
pop = raw['coupon_code'].notna() & (raw['coupon_code'] != '') & (raw['coupon_code'] != 'NaN')
print(raw.loc[pop, 'source_system'].value_counts().to_string())

source_system
JSON    1051
XML     1048


In [46]:
p = product_reviews_final
print(p['contains_non_latin_script'].value_counts().to_string())
print('\nlatin_analysis sentinel rows:', int((p['review_body_latin_analysis'] == 'NaN').sum()))
print('extracted_order_reference populated:', int((p['extracted_order_reference'] != 'NaN').sum()))
print('extracted_product_sku populated:', int((p['extracted_product_sku'] != 'NaN').sum()))

contains_non_latin_script
False    6729
True      271

latin_analysis sentinel rows: 0
extracted_order_reference populated: 7000
extracted_product_sku populated: 7000
